# 042 NWCK Distance in the tree
## グラフにおける木の定義
- グラフが閉路を持たず、連結である。
- 連結なグラフの位数（点数）が n のとき、辺数が n-1 となる。
- 閉路を持たないグラフに隣接していない2点の間にどのように辺を加えても、閉路が1つだけできる。
- グラフの辺を1本でも消すと連結ではなくなる。
- グラフのどの2点間の道は1通りしかない（たどり方は必ず1通り）

- 木の任意の点から任意の点までの道筋は1通りであるため、その道筋の辺の長さを計算すれば、木の2点間の距離が求められる

## Newick format について
- 木をテキストで表記する手法の一種,系統樹などに用いられる
- 終わりは必ず; で示す
- 二分グラフよりも効率が良い。なぜなら、中間のノードを圧縮して表記するから
- 同字中間ノードに接続する葉ノードをneighbers という
```
    R
   / \
  U   D
 / \
A   B
```
- 葉ノードA と B は　内部ノードUのneighbors である
- 従って、((A, B)U, D)R; とくくることができる
- 中間ノードに名前がない場合(Uに名前がついていない場合)は省略可 (A, B) と書く
- 無根系統樹の場合、どの内部ノードを根とするかは自由に決められる。
- 根によってNewkick format は異なる

## Given
- 200個以下のノードで構成された40個以下のツリーが与えられる。
- 各ツリーについて、ツリー内の点 X, Y が与えられる。

## Return
- 一つの辺の長さを1 として、各ツリーとそれに含まれるノードペア間の距離を求めよ

## 考え方
- 共通祖先までたどり着くと( (Aを子に含む枝), (Bを子に含む枝) ) となる
- それ以上外に階層が存在していても考える必要はない
- つまり、A - B 間に存在する () の数を数えればよい

((A, B), (C, D))


In [ ]:
## ワード1とワード2 に挟まれる , (, ) を見れば何個層を上がる, 下がるかを考えることができるのでは

import re

def calc_tree_dist(word1, word2, tree):
    # ワード1とワード2の位置を見つける
    word1_idx = tree.index(word1)
    word2_idx = tree.index(word2)

    # ワード1とワード2の間の文字列を取得
    between_str = tree[min(word1_idx, word2_idx):max(word1_idx, word2_idx)]

    symbols = ["(", ")", ","]
    # ワード1とワード2の間の文字列から、括弧とカンマだけを抽出
    between_brackets = "".join([c for c in between_str if c in symbols])    
    
    # クエリに関係ない閉じた subtree は削除できる
    # ( 任意の文字 ) というパターンでは、((,), (,)) こういうパターンで壊れる
    # だから、再帰的に、( 任意の数のカンマ ) というパターンを除いていく
    while re.search(r'\(,*\)', between_brackets):
        between_brackets = re.sub(r'\(,*\)', '', between_brackets)
    
    # ここで、,),),), (,(,(, というパターンのみになる (クエリに関係ないサブの枝はすべて消える)
    # あとは、(の数を数えればよい
    # パターンとしては, 親-子, 子-親, 子-子　の3パターンがある
    # 親-子, 子-親の 場合は +2 する必要ない 

    # 例えば、(,(,(, というパターンは、親-子
    if between_brackets.count('(') == len(between_brackets):
        return between_brackets.count('(')
    if between_brackets.count(')') == len(between_brackets):
        return between_brackets.count(')')
    if between_brackets.count(',') == len(between_brackets):
        return 2
    return between_brackets.count('(') + between_brackets.count(')') + 2

def get_answer(input_txt):
    lines = input_txt.strip().split("\n")
    # 3行ごとに, tree, word1 word2, 空行 というパターンで入力が与えられる
    answers = ""
    
    for i in range(0, len(lines), 3):
        tree = lines[i]
        word1, word2 = lines[i+1].split()
        dist = calc_tree_dist(word1, word2, tree)
        answers += str(dist) + " "
    
    return answers

In [ ]:
input_txt = """((Accipiter_grupus,Dendrobates_veredus),(((Acipenser_javanica,(Furcifer_torquatus,Phrynohyas_leucogeranus)),(((((Alaus_subcinctus,(Hemitheconyx_albigula,Underwoodisaurus_carolinensis)),(((((Bos_elaphus,Kassina_cavimanus),(Castor_subminiatus,((((Ctenosaura_gallinago,Uroplatus_caninus),Ctenosaura_subminiatus),Paramesotriton_nyroca),Numenius_ibera))),Columba_angulifer),(Cyclagras_exquisita,Theloderma_galactonotus)),Psammophis_monorhis)),Coleonyx_Jankowskii),(Bombyx_versicolor,Rufibrenta_flava)),((((((Ambystoma_javanica,Vulpanser_caeruleus),((Cyclagras_colubrinus,Ziphius_uluguruensis),Cyclagras_means)),Salamandra_multifasciata),Tetrao_blythi),Balaena_tatarica),Psammophis_viridis))),((((Bombina_morinellus,((Lystrophis_falcinellus,sibiricus_arcticus),Mochlus_tanganicus)),Pseudorca_verrucosus),Strepsilas_fulva),Pachytriton_leucomelas)),(((((((((Agama_teguixin,(((Amphiuma_colchicus,Scorpio_ochropus),((((Anas_dahurica,Cynops_rutilans),(((Aythia_caesius,(Coenobita_arseniavi,Ziphius_altaicus)),(Dahurinaia_semipalmatus,(Euspiza_quinquetaeniata,Ninox_tuberculosus))),Nhandu_kuhli)),Ardea_smithii),((((Arctomys_cyanochloris,(((((Boiga_walti,Casarca_bonasus),Chelus_leucophtalmos),Natriciteres_moschiferus),Nerodia_auriculatus),Tylototriton_dolosus)),(Branta_piscator,Columba_geyri)),Hirundo_resinifictrix),Podoces_fallax))),Eschrichtius_irregularis)),(Aythya_timidus,Strepsilas_hodgsoni)),(Epipedobates_standingii,(Hadogenes_occitanus,(Marmota_brevipes,Oligodon_savignii)))),Megaloperdix_czerskii),Holaspis_felderi),Spizaetus_cherrug),Cottus_vanellus),(Allactaga_sibirica,((((Bradypterus_rosmarus,((Ciconia_ferruginea,((Fuligula_cristatella,(Melanocorypha_scalaris,Siniperca_irregularis)),Kaloula_lutra)),Psammophis_ammon)),Thamnophis_gallicus),Lanius_macularius),Crocodylus_fulva))),Chalcides_ignicapillus));
Uroplatus_caninus Hadogenes_occitanus

(Acanthosaura_insularis,Accipiter_monedula,Alcedo_seemani,Antaresia_brevirostris,Aplopeltura_interiorata,Aythya_gibbosus,Bombyx_constricticollis,Buthacus_dentatus,Candoia_monachus,Capella_smithii,Cervus_maurus,Charadrius_dahurica,Chlamydosaurus_rudicolis,Chondropython_mehelyi,Chrttusia_lavaretus,Ciconia_ochropus,Coenobita_squamatus,Corallus_caudatus,Cottus_duplus,Cynops_reinwardti,Dasypeltis_vitulina,Dendrelaphis_subrubrum,Diomedea_schreibersi,Dryobates_canorus,Egretta_exanthematicus,Equus_similis,Eschrichtius_clypeatus,Ethmostigmus_cocincinus,Eublepharis_tadorna,Eurynorhynchus_deserti,Euspiza_sujfunensis,Felis_plumipes,Fulica_marmoratus,Fuligula_naumanni,Furcifer_caesius,Gallinago_aristotelis,Gallinago_hendersoni,Gazella_hodgsoni,Gekko_argali,Geochelone_colubrinus,Gerrhosaurus_anser,Gonocephalus_mlokosiewiczi,Gyps_glareola,Gyps_monedula,Haliaeetus_fulva,Haliaeetus_scalaris,Haliaetus_cambridgei,Halichoerus_tristis,Hyla_fischeri,Ingerophrynus_celeber,Kinosternon_leucomystax,Lamprolepis_vulgaris,Lanius_reinwardti,Larus_pedo,Leiocephalus_chuatsi,Lyrurus_belliana,Melanocorypha_casualis,Micropalama_fuscus,Natriciteres_gallinago,Nerodia_caninus,Nyctixalus_cenchria,Oceanodroma_albopillosum,Osteopilus_parreyssi,Oxyura_major,Parus_hungaricus,Philomachus_mykiss,Podoces_valliceps,Psammophis_aegyptia,Pseudemys_fuscus,Pusa_pulcher,Regulus_melanostictus,Rosalia_glaucescens,Rufibrenta_cepediana,Saga_leucoryphus,Saiga_uncia,Salmo_falcipennis,Salvelinus_pendulinus,Sceloporus_govinda,Scolopax_ciliatus,Sitta_fuscatus,Spalax_grandis,Spalerosophis_aceras,Squaterola_quinquetaeniata,Sterna_melanostictus,Synthliboramphus_calamita,Tringa_belliana,Vanellus_oenanthe,Vulpes_heudei,Ziphius_musculus)Abantias_nipalensis;
Oxyura_major Gallinago_hendersoni

(((((((Agama_franckii,Rhabdophis_platycephala),Eschrichtius_davidiana),Pterinochilus_altaica),Grampus_oxycephalum),((Caiman_colchicus,Paramesotriton_reticulatus),Desnana_cyanus)),(Almo_occitanus,((((Bradyporus_resinifictrix,(Callipogon_anachoreta,Kinixys_lividum)),(((Cervus_tinctorius,((Eublepharis_barroni,Salvelinus_caesius),Neolycaena_undulata)),(Gavia_borealis,Numenius_anachoreta)),Recurvirostra_labiatus)),Pusa_arenarius),Rhacodactylus_calamita))),Alectoris_valliceps,(((((((((((((((((((Allactaga_mandarina,(Trachemys_jubata,Xenopeltis_albocinctus)),((Casarca_alpinus,Enhudra_jubata),(((Cynops_fimbriatus,Remiz_acanthinura),Megaptera_anatina),Ingerophrynus_borealis))),Regulus_calamita),((((((Alopex_moschata,(Enhydris_personata,Psalmopoeus_sudanensis)),Pyrrhocorax_uncia),Scolopendra_horridum),Otocoris_alcinous),Ciconia_venulosa),Tetraogallus_sibirica)),Eirenis_glottis),(Otocoris_giganteus,Platalea_govinda)),(Capra_sieboldii,(Emberiza_pugnax,Porphyrio_nivalis))),((Brachyramphus_seemani,((((Bradypterus_musicus,(((((Circaetus_difficilis,Hyperoodon_taezanowskyi),Telescopus_multituberculatus),Latastia_holbrooki),Scincus_gregaria),Citharacanthus_moschata)),Lystrophis_ampullatus),(Epicrates_insularis,Plegadis_davidiana)),Sceloporus_laticauda)),(Felis_bellii,(Hemitheconyx_rufodorsata,Vulpes_peregrinus)))),(Budytes_helena,Cyriopagopus_arseniavi)),((((Branta_shadini,Gecarcinus_leucoptera),Damon_indicus),Odobenus_jaculus),Phalacrocorax_situla)),Rhinolophus_albopillosum),Hyla_shadini),Pituophis_pugnax),Lycodon_govinda),((((Gonyosoma_marinus,Plethodon_hassanica),Vipera_brachydactyla),Mylopharyngodon_rufodorsata),Turdus_subniger)),(Dyscophus_garullus,Polypedates_adspersus)),(Capra_wogura,Rhacodactylus_falcipennis)),Mareca_gigas),Eurynorhynchus_maihensis));
Lystrophis_ampullatus Telescopus_multituberculatus

(((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((sibiricus_leucorodia)Zosterops_altaicus)Ziphius_opimus)Xenochrophis_sepsoides)Xenochrophis_maihensis)Underwoodisaurus_ferox)Trachemys_pulcher)Thecla_nigrolineatus)Syrrhaptes_calligaster)Synthliboramphus_franckii)Streptopelia_galeatus)Sternotherus_exanthematicus)Selenocosmia_mirabilis)Scolopendra_rupestris)Scolopax_limosa)Ruticilla_sagrei)Rhabdophis_africanus)Rangifer_dendrophila)Pyrgilauda_sirtalis)Pterocles_melanuroides)Porzana_emilia)Pica_subminiatus)Phrynosoma_coturnix)Phelsuma_morinellus)Pelodytes_conicus)Paraphysa_pachypus)Paradoxornis_tadorna)Pagophila_lineatus)Pagophila_licin)Oxyura_prasina)Nyroca_smithii)Nucifraga_sepsoides)Nipponia_zonata)Neophron_valliceps)Myotis_squamatus)Mochlus_sarasinorum)Mochlus_falcinellus)Mareca_albigula)Lycaenopsis_walti)Lutra_botnica)Leuciscus_dispar)Leptobrachium_ruthveni)Lanius_platycephala)Lampropeltis_bobac)Kaloula_temminskii)Hydrosaurus_scincoides)Heterodon_czerskii)Grus_carinatus)Gonyosoma_marmorata)Gazella_rusticolus)Gambelia_africanus)Gallinago_subglobosa)Furcifer_trigonopodus)Furcifer_schokari)Fulica_taczanowskius)Fulica_subcinctus)Eulabeia_mysticetus)Eudrornias_nasicus)Enhydris_valliceps)Enhydris_hispida)Emydura_pulchra)Dendrelaphis_ciliatus)Cygnopsis_garmani)Cygnopsis_amboinensis)Corallus_dominus)Cinclus_relictus)Chrysopelea_mutabilis)Chrysemys_scalaris)Buteo_ampullatus)Bombina_dolosus)Balaenoptera_ladogensis)Apodora_fasciolata)Antaresia_gebleri)Anser_eremita)Androctonus_bifasciatus)Alopex_garullus)Alectoris_haliaetus)Alcedo_brevirostris)Alauda_ampullatus);
Syrrhaptes_calligaster Corallus_dominus

((((((((((((((((Acanthogonatus_vastus,Chlamydotis_glottis),(((((((Amphiuma_hispida,Thymallus_nebularia),(Philacte_stimsoni,Porzana_interiorata)),(((((Capeila_fulva,Cinclus_proteus),Turdus_shadini),Tadarida_longipes),Cottus_compactus),(Pachydactylus_lepturus,Rhabdophis_nebularia))),Osteopilus_constricticollis),(((Chrysopelea_fissipes,Latastia_hipposideros),Geochelone_cancerides),(Lyrurus_stylifer,(Pusa_soloensis,Salamandra_zenobia)))),(Eryx_mysticetus,((Perdix_duplus,sibiricus_mandarina),crecca_marmorata))),(Dendrelaphis_fluviatilis,Nhandu_edulis))),(Pelodytes_tigrinus,Sitta_fasciolata)),Ptychozoon_morinellus),Synthliboramphus_tricolor),(((((((((Balaenoptera_eulophotes,Capella_chinensis),Scorpio_opimus),(Capreolus_sibirica,Cottus_geyri)),(((Basiliscus_carnifex,Damon_doctus),(Ceratophrys_tinnunculus,Uncia_anachoreta)),((Euspiza_aspera,Odonthurus_Anas),Grampus_mandarina))),Dipus_septentrionalis),Platemys_enydris),Brachyramphus_citreola),Limnaeus_glottis),Cyclagras_paganus)),Platalea_fuscus),Moschus_subglobosa),Rissa_paradisi),Leuciscus_guentheri),(Gambelia_gebleri,(Grampus_torquata,Petrocincla_elaphus))),Recurvirostra_dexter),Cynops_geniculata),Sphenurus_striatus),(((((((((((((Ameiva_spinifera,Uromastyx_intermedia),Middendorffinaia_mutabilis),Eublepharis_moschata),Bradyporus_ruthveni),Cyclagras_arseniavi),Dyscophus_maurus),Totanus_torquata),Gonyosoma_bicoloratum),(((((Coregonus_longicaudata,Dasypeltis_equestris),Neolycaena_morinellus),Megophrys_campestris),Eudramias_ocellatus),Squaterola_caniceps)),Leptopelis_nigriceps),(Ctenotus_horridum,Litoria_mycterizans)),(Chalcides_dauricus,(Laudakia_notaeus,Salvelinus_pygmeus))),(Androctonus_leucogaster,Iomachus_caucasicus))),((((((Acanthoscurria_battersbyi,Corallus_coelestis),Pachydactylus_dominus),Leptopelis_crocodilus),Rufibrenta_cyanochloris),Xenopeltis_quinquestriatus),Totanus_fragrans),((((Acanthoscurria_fuscatus,(Corvus_doriae,Kinosternon_classicus)),(Caiman_mystaceus,(Hadogenes_triangulum,Porphyrio_argali))),((Colaeus_leucoptera,Ctenosaura_irregularis),Spermophilus_limosa)),Brachyramphus_carinatus));
Basiliscus_carnifex Totanus_torquata

(crecca_cachinans,Zosterops_tristis,Vulpanser_shadini,Ursus_scripta,Ursus_mnemosyne,Ursus_Anas,Tylototriton_eximia,Trionyx_lutris,Teratolepis_javanica,Teratolepis_angustirostris,Selenocosmia_hasselquistii,Selenocosmia_arcticus,Scolopendra_barroni,Scolopax_decor,Salvelinus_hassanica,Salmo_mycterizans,Rissa_tarda,Rhodostethia_guttata,Rhodostethia_corsac,Pterocles_tentaculatum,Platalea_wogura,Phrynohyas_ibis,Phrynohyas_cinclus,Phrynocephalus_acutus,Pelodytes_caesius,Parus_serricollis,Paramesotriton_stellio,Pachydactylus_scabra,Ovis_imperator,Ovis_erythrogastra,Otocoris_boa,Ophisops_multifasciata,Opheodrys_chamaeleontinus,Odobenus_cinereus,Nyroca_Bernicla,Numenius_heliaca,Net_adamsii,Neolycaena_wogura,Neolycaena_leucomelas,Nemachilus_notaeus,Megophrys_weliczkowskii,Leuciscus_stellatum,Leiopython_wislizeni,Leiopython_quinquestriatus,Lanius_getula,Lamprolepis_heudei,Lagenorhynchus_meermani,Hyperoodon_chukar,Hemitheconyx_weliczkowskii,Grus_aspera,Grampus_dubius,Grammostola_amboinensis,Gonyosoma_ichthyaetus,Gongylophis_castaneus,Gavia_grus,Gambelia_leucorodia,Eurynorhynchus_querquedula,Equus_hendricksoni,Emydura_odoratus,Emberiza_leucoryphus,Diomedea_laticauda,Damon_rostratus,Cyclagras_albopillosum,Cuon_kraepelini,Coturnix_brevipes,Corvus_salvator,Columba_deminutus,Chrysemys_longicaudata,Chrttusia_marcianus,Chlamydosaurus_terrestris,Capella_infrafrenata,Canis_melanoleucus,Camptoloma_veredus,Callipogon_clypeatus,Buteo_dendrophila,Bombyx_felderi,Bombycilla_azureus,Argynnis_alba,Apalone_caryocatactes,Anodonta_bonasus,Allobates_albigula,Allactaga_ferina,Agama_alpestris,Aegypius_subrufa,Aegialifes_cristatella,Actitis_smithi,Acipenser_musicus)sibiricus_maculatum;
Grammostola_amboinensis Pachydactylus_scabra

((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((crecca_cinerea)Ziphius_garulla)Uroplatus_labiatus)Tursiops_pica)Totanus_circia)Terpsihone_fusca)Tadorna_anser)Tadarida_personata)Sus_serpentina)Sternotherus_lopatini)Sterna_clinatus)Salmo_valliceps)Salmo_crispus)Ruticilla_alcinous)Rhynchophis_radiata)Rhynchaspis_aspera)Rhodostethia_paradoxus)Rangifer_radiata)Querquedula_carnifex)Ptyodactylus_irregularis)Psammophis_acutus)Psalmopoeus_mexicanum)Psalmopoeus_auriculatus)Pratincola_melanuroides)Polypedates_griseus)Philothamnus_maihensis)Phasianus_flava)Petrocincla_cristatus)Pelodytes_dentatus)Pedostibes_caudicinctus)Passer_totanus)Ophisops_hyemalis)Nyctixalus_dahurica)Nucifraga_duplus)Norops_avinivi)Nipponia_leucophtalmos)Net_paradisi)Myotis_paganus)Mylopharyngodon_gallinago)Monodon_fallax)Madagascarophis_bairdii)Limnodromus_bairdii)Lasiodora_glutinosus)Lamprolepis_bairdi)Kinosternon_taczanowskius)Iomachus_vittatus)Iguana_marinus)Holodactylus_monoceros)Hirundo_eremita)Hemiscorpius_grossmani)Halichoerus_perdix)Grampus_compactus)Gonyosoma_mystacinus)Gonyosoma_guineti)Glareola_Jankowskii)Gerrhosaurus_campestris)Gavia_licin)Felis_comicus)Falcipennis_maldivarum)Eurynorhynchus_hassanica)Eublepharis_multifasciata)Eubalaena_iankowskii)Eschrichtius_pholeter)Erpeton_fuliginosus)Erpeton_cyanogenys)Erpeton_constricticollis)Equus_lepturus)Enhydris_glareola)Enhydra_glottis)Elseya_regius)Dryobates_triangulum)Diomedea_lopatini)Dahurinaia_cenchria)Cyriopagopus_hipposideros)Cygnus_gallinago)Cyclemys_fuliginosus)Cuora_caudata)Corallus_bicinctores)Columba_odoratus)Citharacanthus_serpentina)Cinclus_taxus)Chelydra_maldivarum)Chelydra_eremita)Chelus_elaphus)Certhia_torquatus)Brachyramphus_bicinctores)Boa_edulis)Arctomys_horridum)Apodora_flavomaculatus)Anthropoides_zagrosensis)Amphiuma_capreolus)Ameiva_scabra)Almo_picta)Alectoris_maritimus);
Equus_lepturus Iomachus_vittatus

(Androctonus_semipalmatus,((((((((((Apalone_spilota,Eulabeia_garmani),(Glareola_monilis,Phormictopus_quinquetaeniata)),(Pleurodeles_mystaceus,Xenophrys_paradisi)),(Gazella_altaica,Nemorhaedus_crucigera)),Spalerosophis_fernandi),Hadogenes_resinifictrix),((((((((((((((((Argynnis_nigropalmatus,((Crotaphytus_giganteus,Felis_similis),Leuciscus_calligaster)),Pusa_mysticetus),Ptychozoon_casualis),(Nyctixalus_aleutica,Pelomedusa_zagrosensis)),(Castor_calligaster,Pelecanus_acanthinura)),Tamias_ruthveni),Saiga_taczanowskius),(Kinixys_ignicapillus,Pelusios_sarasinorum)),(Ovis_mexicanum,Phelsuma_caryocatactes)),Lampropeltis_taxus),((((Callipogon_aristotelis,Trapelus_trianguligerus),(Moschus_fluviatilis,Phormictopus_peregrinus)),Vulpanser_leucomystax),((Gyps_middendorffi,Limnaeus_hypoleucos),Kinixys_atra))),(((Chrysemys_gemmicincta,Rhombomys_buccata),Hadrurus_gallicus),((Mogera_garmani,Philomachus_caudata),(Otocoris_ibera,Pica_eulophotes)))),(Dendrobates_strepera,Rangifer_taeniura)),Parus_triangulum),Theloderma_deserti),((((((Bos_melonotis,Syrrhaptes_horridum),Ninox_relictus),(Eulabeia_nebularia,Zosterops_leucotus)),Furcifer_melanuroides),Procellaria_lesueurii),Callipogon_veredus))),Pratincola_gallinago),Pelomedusa_semipalmatus),Dipus_odoratus),(((((Arctomys_scripta,((Arenaria_enhydris,Hirundo_rubicola),((Larus_maculata,(Mabuya_Jankowskii,Opheodrys_pholeter)),Ursus_indica))),(((((Calotes_atriceps,Netta_dignus),(Scorpio_mandarina,Sus_gordoni)),Elseya_cranwelli),(Net_dexter,Oceanodroma_fissipes)),Ruticilla_musculus)),Calotes_chuatsi),((((Ceratophrys_carbonaria,Paramesotriton_dignus),Dryobates_cambridgei),(Dipsosaurus_blakistoni,((Litoria_vermiculatus,Marmota_noctua),Notophthalmus_isabellina))),(Rhacophorus_leuconotus,Thymallus_turtor))),Buteo_septentrionalis));
Net_dexter Larus_maculata

((((((((((((((Acanthis_radiata,Cuon_armeniacus),Capra_cliffordii),((((Crotaphytus_flavolineata,Cuculus_sepsoides),Pelomedusa_quadrivirgata),Phrynohyas_peregusna),Vulpanser_bobac)),Madagascarophis_pardus),Theloderma_dennysii),Babycurus_interpres),Capra_fasciata),((Aphonopelma_bobac,(Circus_dexter,Grampus_quadriocellata)),Bubulcus_cavirostris)),((((((Argynnis_chuatsi,Eryx_solitaria),((Damon_coelestis,Podoces_tristis),Paramesotriton_gibbosus)),(Oenanthe_aegyptia,(Phrynops_monilis,Spalax_porzana))),Melanocoryhpa_rosea),Paramesotriton_climacophora),Cuculus_femoralis)),((((((((Anas_turtor,((((Arctomys_rufodorsata,Colaeus_sauritus),Dahurinaia_haliaetus),(((Ciconia_himantopus,(Iomachus_casualis,Totanus_ignicapillus)),Ptyodactylus_alterna),((Lamprolepis_smaragdina,Parus_aegyptia),(Machetes_Jankowskii,(Pelomedusa_virgo,Pituophis_salei))))),Nemorhaedus_cristatellus)),Tetraogallus_africanus),Pandion_sagrei),Hemitheconyx_pictus),((((((((Balaena_baibacina,((Platalea_rhymnus,Teratolepis_leucostomum),Tringa_melonotis)),Falcipennis_bengkuluensis),Nyctixalus_drapiezii),Hydrochelidon_altaica),Ursus_musicus),(((Bubulcus_minor,(((((Corallus_cristatus,Xenochrophis_hardwickii),Phasianus_saxatilis),((Cygnus_jaspidea,Porzana_turtur),(Dafila_ichthyaetus,Hyla_gibbosus))),(((Diomedea_platycephala,Ethmostigmus_rusticolus),Rufibrenta_querquedula),Euspiza_docilis)),Hottentotta_gecko)),Lagenorhynchus_latiscutatus),((Coregonus_triangulum,(Net_ulikovskii,Vormela_cristatus)),Tiliqua_melleri))),((Gypaetus_mlokosiewiczi,Nyroca_ferruginea),Turdus_hyemalis)),Cinclus_monoceros)),Pyrgilauda_perdix),Leiocephalus_cherrug),Chrttusia_ochropus)),Eudrornias_carnifex),((Androctonus_avinivi,(Athene_alba,Myotis_subglobosa)),Anodonta_canorus)),((Eryx_leucoryphus,Ketupa_melanostictus),Python_fernandi)),(Alauda_chamaeleontinus,Grammostola_moschiferus),Alpes_oedicnemus);
Rufibrenta_querquedula Coregonus_triangulum

(Aix_hassanica,(((((((((((((((((((((Aix_scripta,(((Boiga_ferrumequinum,((Chelodina_anatina,(Ethmostigmus_aspera,Tropidurus_colombianus)),Tetrao_glareola)),Cygnus_carbo),Cuora_dactylisonans)),((((Burhinus_hemilasius,Lutra_cocincinus),((Egretta_getula,Thymallus_castus),Leiurus_erythropus)),Corvus_tatarica),Phrynocephalus_paradisi)),Almo_armeniacus),(Gonyosoma_ceterus,(Grampus_opimus,Spizaetus_dactylisonans))),((((Balaenoptera_arcticus,Chen_fulvus),(Cervus_bimaculata,(Columba_pugnax,Spizaetus_hyemalis))),Nyctaalus_schrencki),Dafila_leucogeranus)),(Euspiza_macrops,(Pelomedusa_temminskii,(Physignathus_fallax,Psalmopoeus_fallax)))),(Anthropoidae_boyciana,Cygnopsis_perrotetii)),(Halichoerus_salvator,Porzana_perrotetii)),(Mylopharyngodon_gallicus,(Paraphysa_flava,Scincus_alpina))),(Kassina_chuatsi,Lycaenopsis_bairdii)),Sus_moschiferus),((Dipus_brevirostris,(Falcipennis_arizonensis,Hydrochelidon_placidus)),Ethmostigmus_septentrionalis)),Anodonta_kraepelini),Passer_caucasicus),(Desnana_situla,Iguana_mlokosiewiczi)),Opheodrys_turneri),(Anthropoidae_thibetanus,((Bufo_guttata,Panthera_cocincinus),(Haplopelma_scrofa,Oenanthe_spilota)))),(Epicrates_dentatus,Tryngites_tarandus)),Melanocorypha_mycterizans),((Gonocephalus_buccata,Perdix_licin),Rosalia_alpestris)),((Asthenodipsas_peregrinus,Recurvirostra_platycephala),Tringa_savignii)),((Alaus_africanus,(((((Bombus_moschata,((((Moschus_dispar,Picus_flavolineata),Phrynohyas_calvus),Tadorna_personata),Streptopelia_giganteus)),Pareas_prasina),Capella_tatarica),((((Brachyramphus_hypoleucos,(Capra_marcianus,Middendorffinaia_ferruginea)),Querquedula_rutilans),Grus_cornix),Callipogon_albocinctus)),((Bombyx_scalaris,((Lycaenopsis_leucocephala,Madagascarophis_avinivi),Vipera_comicus)),(Philomachus_getula,Thecla_canus)))),((((Aythia_quadrivirgata,(Fulica_guangxiensis,Nerodia_mirabilis)),Haliaetus_sibirica),((Calidris_caudata,Coenobita_scalaris),Otis_timidus)),(Kassina_fuscus,Marmota_Jankowskii))));
Almo_armeniacus Nerodia_mirabilis

(((((Accipiter_godlewskii,(Podoces_tigris,Sturnus_jubata)),Spalerosophis_alba),Capra_davidiana),(Aquila_schneideri,Monodon_brongersmai)),((((Actitis_nupta,((((((Dafila_proteus,Hydrosaurus_aegyptia),Eulabeia_azureus),Gerrhosaurus_cepediana),Fulica_pygmeus),Melanocoryhpa_leucoryphus),Lampropeltis_hispida)),Cuculus_canorus),(Desnana_falcipennis,Xenopeltis_citrsola)),Perdix_personata),(((((((Alaus_alpinus,Hadogenes_xanthocheilus),(((Athene_dauricus,Larus_monoceros),((Leptobrachium_metallica,Neophron_ovata),Ruticilla_bicoloratum)),Colaeus_obsoleta)),Rhacodactylus_pulcher),((((((((((Anolis_doctus,(Phalacrocorax_auratus,Pleurodeles_caerulea)),((Clemmys_stagnalis,Trionyx_helena),Leiolepis_cianeus)),Arctomys_ovata),Micropalama_margaritifera),Elseya_subglobosa),(Kinosternon_caudolineatus,Siniperca_schokari)),(Net_querquedula,Ovis_lobatus)),Osmoderma_flammea),Argynnis_trigonopodus),Marmota_isabellina)),((((((((((Aphonopelma_arcticus,Caiman_heliaca),((((Eryx_capreolus,(Iguana_climacophora,Platemys_stellatum)),Phelsuma_schneideri),(Rosalia_ignicapillus,Telescopus_melleri)),Saga_triangulum)),((((Apodora_zenobia,((Aquila_teguixin,(Macrorhamphus_porzana,(Monodon_euptilura,Nemachilus_flavomaculatus))),Oceanodroma_siebenrocki)),(Lystrophis_gregaria,Phrynosoma_schneideri)),Pseudorca_temminskii),Felis_caudolineatus)),Sterna_laticauda),((Aythya_monilis,((((Bombycilla_fissipes,Casarca_doctus),(Falco_garulla,(Hirundo_smaragdina,(((Pogona_collectivus,Trachemys_breitensteini),Pseudemys_mysticetus),Rufibrenta_saiga)))),Dahurinaia_grupus),(((Chelus_hardwickii,Dendrelaphis_weliczkowskii),Lyrurus_leuconotus),Psalmopoeus_fulva))),Ctenosaura_dorsalis)),((((((Capeila_bonasus,Desnana_semipalmatus),Pterinochilus_hosii),Hirundo_cynodon),Testudo_femoralis),(Petrocincla_sepsoides,Salvelinus_gallicus)),Hadogenes_seemani)),Arenaria_borealis),Chlamydosaurus_celeber),(Gongylophis_alpinus,(Lamprolepis_blakistoni,Pseudorca_calidris))),Odobenus_calamita)),Antilope_perdix),Gerrhosaurus_felderi));
Caiman_heliaca Hydrosaurus_aegyptia

(,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,Saga_hemionus,,Sus_septentrionalis,,,,,,,,,,,);
Sus_septentrionalis Saga_hemionus

((((Abantias_angustirostris,(Burhinus_clinatus,Coregonus_ignicapillus)),((((((((((((((Allactaga_paradoxus,(Osteopilus_chrysaetus,Rhombomys_nivalis)),Motacilla_caelebs),(Hadogenes_angustirostris,Pagophila_stellio)),(((((Androctonus_standingii,((((Anodonta_vulgaris,(Chrttusia_cenchria,Dasypeltis_calyptratus)),Babycurus_torquatus),((((((Athene_maurus,Bombina_graculus),Budytes_chamaeleontinus),Aythya_monacha),Rufibrenta_hemionus),Pleurodeles_glacialis),Podoces_schneideri)),((Marmota_pendulinus,(Mergus_bimaculata,Scaphiopus_flavigularis)),Pelodiscus_fallax))),Python_marmoratus),((Fregilegus_temminskii,Mabuya_chrysaetus),Tupinambus_brevirostris)),(Bronchocela_acanthinura,Rufibrenta_cranwelli)),(Archispirostreptus_jaspidea,Buthus_metallica))),Polypedates_ridibundus),((((((((((((Allobates_botnica,Pelodiscus_euptilura),((((((((Ambystoma_yeltoniensis,((Ardea_unicolor,(Hottentotta_wislizeni,(Osmoderma_glottis,Uncia_zenobia))),Underwoodisaurus_sauritus)),(Eunectes_cancerides,Lampropeltis_ladogensis)),Porphyrio_pallasii),Colaeus_hemilasius),Thymallus_albertisii),(((Haliaeetus_sauromates,Latastia_ampullatus),(Mylopharyngodon_godlewskii,(Tringa_mystacinus,Zosterops_gobio))),Phrynocephalus_maritimus)),Hottentotta_cornix),(Eunectes_piscator,Pachytriton_opimus))),((Alopex_lavaretus,Eryx_getula),(Eumeces_lutra,Neophron_ceterus))),Nemorhaedus_homeana),Tylototriton_pholeter),(((Bronchocela_arcticus,(Leiocephalus_boulengeri,Tamias_ornata)),((Erpeton_aegagrus,Eumeces_crispus),Pelodiscus_graeca)),Damon_capreolus)),Calidris_plumipes),Osmoderma_maculata),(Leiocephalus_fuliginosus,(Pterinochilus_hassanica,Vulpes_spinifera))),Citellus_tetrix),(Lycodon_sujfunensis,(Mochlus_cioides,Saga_paradisi))),Anser_indica)),Petrocincla_pallidus),((Anas_ion,Squaterola_meermani),Leuciscus_cyanus)),Chlamydotis_dendrophila),(((((Chelus_taxus,(Madagascarophis_middendorffi,Ptychozoon_grus)),Colaeus_kazanakowi),Cyclemys_bonasus),Pareas_irregularis),Trionyx_auriculatus)),Marmota_arizonensis),Cynops_versicolor),Picus_quadrivirgata),Gambelia_nippon)),Chettussia_dendrophila),Acanthoscurria_tigris,Actitis_septentrionalis);
Picus_quadrivirgata Tamias_ornata

((((((((((((((Acanthoceros_cingulata,(((Chelodina_calligaster,Laudakia_pulchra),Morelia_torquata),((Ephibolus_taezanowskyi,Gerrhosaurus_mysticetus),Kinosternon_apollo))),Lutra_rudicolis),Gavia_galericulata),Streptopelia_australis),(Heterodon_purpurascens,Holodactylus_sieboldii)),Bradyporus_holbrooki),Squaterola_trianguligerus),(((((Aix_nebrius,((Elaphe_helvetica,Procellaria_decorus),sibiricus_japonensis)),Morelia_miliaris),((Chen_blythi,Varanus_hasselquistii),Cinclus_avosetta)),((Anthropoidae_sauritus,Vormela_atthis),(Aquila_cinerea,Chettussia_undulata))),(Apus_pachypus,(Carabus_dentata,Emydura_Jankowskii)))),((Eumeces_mirabilis,(Hadogenes_labiatus,Lepidobatrachus_cygnoides)),Sus_isabellina)),(Phyllopneuste_stimsoni,Synthliboramphus_barbata)),((((((((((Anas_cherrug,(Atrophaneura_caudicinctus,(Gonocephalus_melanoleucus,(Pica_pholeter,Varanus_arcticus)))),Liasis_marcianus),(((Callipogon_ion,Meles_viscivorus),Porphyrio_guineti),Oxyura_brandtii)),Gonocephalus_guentheri),Platalea_battersbyi),Bradyporus_garmani),Argynnis_albopillosum),Pedostibes_avocetta),(Chrttusia_piscator,Pelusios_niloticus)),Eubalaena_cliffordii)),Cuculus_salei),Pratincola_boulengeri),Aegypius_haliaetus,(((((((Agama_rostratus,((Athene_apollo,Monticola_hasselquistii),Himantopus_guentheri)),((((Bombus_virgo,(Fregilegus_crassidens,((Lamprophis_scincoides,(Rufibrenta_docilis,Tadarida_hemionus)),Megaptera_morinellus))),(Eunectes_vereda,Ketupa_uluguruensis)),Polypedates_macqueni),Chlamydotis_lutris)),Scolopax_barroni),(Lystrophis_mysticetus,Mustela_stylifer)),(((Bombus_emilia,(Bufo_glacialis,(Chelus_nipalensis,(Gerrhosaurus_arizonensis,Phrynocephalus_tolai)))),Chrttusia_squamatus),Selenocosmia_fusca)),(((((((((Alloporus_bukhunensis,(((Anas_conicus,Gavia_salamandra),Corytophanes_leucoptera),Holodactylus_armeniacus)),Streptopelia_wogura),Neolycaena_glaucescens),((Ketupa_davidiana,Upupa_ladogensis),Nyroca_nigropalmatus)),Chondropython_filipjevi),Pareas_vulgaris),Trionyx_saxatilis),Gypaetus_salei),Gallinago_nigriceps)),(Athene_stellatum,Lamprolepis_rufinus)));
Chen_blythi Lepidobatrachus_cygnoides

((Abantias_rubicola,((((((((((((Ahaetulla_aureola,Epicrates_crispus),(Felis_equestris,Underwoodisaurus_armata)),(((((((((((Anser_ibera,Cyclemys_quadriocellata),Tamias_physalus),sibiricus_ameiva),(Crotaphytus_cygnoides,(Cyriopagopus_thibetanus,(Rhamphiophis_cornix,Turdus_caninus)))),(((((((Basiliscus_flammea,Casarca_corone),Cuculus_ameiva),Corallus_bairdi),Rhesus_kazanakowi),Morelia_savignii),(Ceratophrys_fasciolata,(Mustela_lehmanni,Nemorhaedus_paganus))),Citellus_aestivus)),(((((((Arenaria_edulis,((Chlidonias_bicinctores,Mogera_sagrei),sibiricus_equestris)),(((Leptopelis_terrestris,Osteopilus_albopillosum),Mabuya_tentaculatum),Polypedates_pallasii)),Pyrgilauda_noctua),(Fuligula_nipalensis,Mylopharyngodon_cyanochloris)),Oxyura_vulpes),Eudramias_constrictor),((Himantopus_hypoleucus,Notophthalmus_squaterola),Strepsilas_perrotetii))),Fregilegus_arcticus),(Balaena_siebenrocki,(Lamprophis_marinus,(Streptopelia_mugodjaricus,Xenophrys_drapiezii)))),((Ceratophrys_cinclus,Grammostola_stylifer),(Falcipennis_dahurica,(Mochlus_schokari,Thymallus_gregaria)))),Regulus_subminiatus),Grampus_aleutica)),((((Alectoris_hispida,((Mochlus_melonotis,Physignathus_relictus),Sorex_armeniacus)),Platemys_circia),(((((Bufo_ruficollis,((Holaspis_botnica,Teratolepis_macularius),Lutra_carinata)),Eirenis_tigrinus),Leiopython_means),Paraphysa_subminiatus),(((((Coregonus_stylifer,Osmoderma_vipio),Eudramias_caudicinctus),Pterocles_cingulata),Kaloula_fiber),Egretta_alpinus))),Paramesotriton_insignis)),Hadogenes_nivicola),Tamias_ciliatus),(Grampus_uncia,Opheodrys_strepera)),((Gekko_occitanus,Micropalama_collaris),Psalmopoeus_zagrosensis)),(Eurynorhynchus_leucogaster,Rangifer_querquedula)),(Calidris_quinquestriatus,Chen_rosmarus)),(((Certhia_avocetta,Leptopelis_dactylisonans),Python_eximia),((((Cynops_sebae,Saiga_lutris),Melanocorypha_bedriagai),Trachemys_catenifer),Salamandra_longipes))),(Chrysemys_grus,Falco_fasciata))),((Acanthoscurria_squamatus,(Antilope_nigriceps,(Ptychozoon_ussuriensis,Tetraogallus_pedo))),Scolopax_arvensis),(Accipiter_ruficollis,Gypaetus_olivacea));
Salamandra_longipes Calidris_quinquestriatus

((((((((Agama_coloratovillosum,Circus_uluguruensis),((((((((Anolis_caryocatactes,(Leptopelis_sibilans,Oedura_godlewskii)),(Aplopeltura_calvus,(((Eubalaena_salei,(((Gambelia_dulkeitiana,Spermophilus_scripta),(Limnaeus_tarandus,Tryngites_cristatella)),Thecla_garulla)),Nyctaalus_macrops),Rissa_carbo))),(((((Antaresia_hilarii,Python_nipalensis),Bubulcus_noctua),Psammophis_monoceros),Elseya_edulis),(((Apalone_leucogeranus,((((Cynops_acanthinura,(Telescopus_zonata,sibiricus_australis)),Latastia_nippon),Lasiodora_subrufa),Grammostola_caerulea)),(Fuligula_avinivi,Lepidobatrachus_duplus)),Ruticilla_linaria))),Aplopeltura_decorus),(((Gonocephalus_pugatshuki,(Leiolepis_nigriceps,(((Middendorffinaia_diadema,Ovis_geyri),Paraphysa_leucogeranus),(Neophron_albopillosum,Teratolepis_major)))),Psammophis_cristata),Vanellus_chinensis)),Eubalaena_dentatus),(Enhydra_pulchripes,Sericinus_botnica)),(Capreolus_pendulinus,Philacte_cliffordii))),((Allactaga_kazanakowi,(((Chrttusia_mystaceus,Furcifer_similis),(Felis_tigrinus,Monodon_flavescens)),Thamnophis_zagrosensis)),Pyrgilauda_paradoxus)),(Picus_griseus,Plegadis_monilis)),Rhodostethia_altaicus),(((((((((((Alectoris_yeltoniensis,(Liasis_carbo,Tringa_lutris)),Hysterocrates_leuconotus),Atrophaneura_nigra),Saiga_marmoratus),Aythya_caninus),Balaena_vereda),(Cardiocranius_sujfunensis,Pyrrhocorax_vitulina)),Rhabdophis_hardwickii),Phrynocephalus_macrops),Holaspis_nigrolineatus),Rhamphiophis_crassicauda)),(Gonyosoma_macularius,Lutra_armeniacus)),(Alaus_quadrivirgata,((((Crotaphytus_tadorna,Latastia_vanellus),Oligodon_glaucescens),Laudakia_doriae),Cygnopsis_avosetta)),(((Alectoris_versicolor,(Corallus_circia,(Lyrurus_miliaris,Pyxicephalus_unicus))),Capeila_dispar),Gonocephalus_prasina));
Saiga_marmoratus Gonocephalus_pugatshuki

(((Aegialifes_galeatus,(Fregilegus_venulosa,Tamias_melanuroides)),(((((((((((((Almo_flavescens,(Emydura_chrysaetus,Phalaropus_graeca)),Pusa_hipposideros),Tryngites_fissipes),(Megaloperdix_verrucosus,Morelia_heliaca)),((Balaena_solitaria,Enhydris_hongkongensis),Neophron_verrucosus)),Netta_nebularia),Cyclemys_fuscatus),(Calotes_barroni,Cardiocranius_sp)),((Oedura_daurica,(Pterocles_crispus,Rosalia_eulophotes)),Rhinolophus_dexter)),Podoces_pulchra),Rhodostethia_cygnus),Chondropython_novaeguineae),(((((Argynnis_cavirostris,((((((Aythya_peregusna,Sericinus_shadini),Podoces_prasina),Macrorhamphus_longipes),(((Cyclagras_aeruginosus,(Hemitheconyx_cristata,Phrynocephalus_obsoleta)),Erpeton_plumipes),Dipsosaurus_leucostomum)),(Citharacanthus_bonasus,((((Enhudra_peregrinus,crecca_tarandus),Nemachilus_moschiferus),((Lystrophis_sarasinorum,Neophron_zagrosensis),Pandinus_modestus)),Pogona_montela))),Heterodon_doriae)),Pelusios_garullus),(Dipsosaurus_not,((((Dipsosaurus_zonata,(Lobipes_belliana,Totanus_dignus)),((Eunectes_nivicola,(Pedostibes_veredus,Pelodiscus_borealis)),Tupinambus_subcinctus)),Synthliboramphus_campestris),Eumeces_oxycephalum))),(((((Chlidonias_bicinctores,Notophthalmus_peregrinus),(Cyriopagopus_hassanica,Limnaeus_dominicus)),Nyctixalus_leucoryphus),Elaphe_longicaudata),Riparia_tetrax)),Pleurodeles_strepera))),Aix_livia,(((((((((Alauda_glaucescens,Platemys_medirostris),Buthacus_troglodytes),((Aythya_cambridgei,Procellaria_querquedula),Psalmopoeus_nigra)),Cottus_wislizeni),Pachytriton_auriculatus),Riparia_kopsteini),((((Amphiuma_vereda,Glareola_glutinosus),Moschus_arseniavi),Eucratoscelus_salei),Sturnus_exanthematicus)),Regulus_wumuzusume),((Osteopilus_melanoleucus,Vipera_minor),Uromastyx_barroni)));
crecca_tarandus Enhydris_hongkongensis

((((,(,)),(((((((((,),),),((((((((((,(,(((((,),),),(,)),))),),((,(,(,))),)),),(((((((((((((((,),),),),(,)),),),(((,),),)),),),),(,(,))),),),)),),((Gongylophis_pictus,),)),),),(,Monticola_giganteus))),((((,),(,)),(,)),(((((((,),),((,),(,))),),(,)),),))),),),),)),),,((((((,),),),),),));
Monticola_giganteus Gongylophis_pictus

(((((((((((Acheron_multifasciata,(Cuculus_ridibundus,((Cygnus_bairdii,Pseudorca_coelestinus),(Kinixys_crassidens,Rangifer_ferox)))),Enhudra_hardwickii),(((Chrttusia_cianeus,Lepidobatrachus_pelagicus),(Lycodon_modestus,Thamnophis_sibilans)),(Plegadis_xanthocheilus,Spalerosophis_bewickii))),(((((((((((Aplopeltura_scincoides,Platalea_dulkeitiana),Sceloporus_lavaretus),Kaloula_ochropus),(((Falco_nipalensis,Platemys_ibera),Hyperoodon_tatarica),Saxicola_hirundo)),Macrorhamphus_glaucescens),((((Dasypeltis_erythrogastra,(Furcifer_teniotis,Scaphiophryne_erythropus)),Grammostola_citreola),(((Emberiza_anatina,Ethmostigmus_tuberculosus),Gonocephalus_miliaris),(Kinixys_collaris,Leiolepis_teguixin))),Haliaeetus_piscator)),Larus_agama),(Mesoplodon_kopsteini,Pyrgilauda_eximia)),Dahurinaia_leucocephala),Argynnis_constricticollis),((((Chlamydotis_terrestris,Dendrelaphis_acuta),Spizaetus_leiosoma),Erpeton_giganteus),Uromastyx_notaeus))),Seokia_maihensis),Citellus_notaeus),((((((((((((((Boa_tinnunculus,((((Bradyporus_gordoni,Eulabeia_dispar),Sorex_falcinellus),((Eirenis_chrysaetos,Parnassius_clinatus),Glareola_atrigularis)),Sphenurus_celer)),Haliaeetus_calamita),Cygnopsis_canus),Corvus_ridibundus),Querquedula_brandtii),Uroplatus_serricollis),Damon_variabilis),(Coenobita_eburnea,Gonocephalus_prasina)),Gekko_aristotelis),Pelomedusa_caudata),Python_standingii),Epipedobates_variegatus),Kaloula_taeniura),Vanellus_decor)),Machetes_corsac),Mabuya_oenanthe),Iomachus_opimus),Agama_cristatellus,((((Agama_laevis,Osteopilus_aristotelis),(Nemachilus_aegagrus,Xenophrys_manul)),((((Crocodylus_japonensis,Limnodromus_campestris),(Teratolepis_dexter,Thamnophis_femoralis)),Python_nasuta),(Nyctaalus_oenanthe,Pandion_difficilis))),Chrttusia_cyanogenys));
Platemys_ibera Platalea_dulkeitiana

((Alopex_oxycephalum,Leiurus_atra),((((((((((Amphiuma_sudanensis,(Burhinus_leucogeranus,Pelodiscus_trianguligerus)),Osteopilus_helena),Hadogenes_monoceros),Chondropython_intermedia),(Apalone_mlokosiewiczi,((((Charadrius_erythropus,Oenanthe_soloensis),((((Eublepharis_alcinous,((Salamandra_prominanus,Uncia_spilota),Scorpio_schneideri)),Haplopelma_macularius),((Falco_doctus,(Leiolepis_galactonotus,Phalaropus_interpres)),Melanocoryhpa_albopillosum)),Minipterus_karelini)),((Cinclus_leucogeranus,Middendorffinaia_terrestris),Sus_corsac)),Grammostola_intermedia))),Chrttusia_arenarius),Osmoderma_meermani),(((Asthenodipsas_maurus,((((Brachyramphus_saiga,((Capeila_vermiculatus,Hyla_scrofa),((((Eryx_battersbyi,(Hyla_interiorata,(Litoria_rutila,Monticola_zenobia))),Recurvirostra_vanellus),Trapelus_tetrix),Falcipennis_molurus))),Pleurodeles_Anas),Underwoodisaurus_lineatus),((Cypselus_albigula,Hysterocrates_perdix),Kinosternon_getula))),((((Circus_blakistoni,Eudrornias_leucomelas),(Kinixys_kuhli,Leiolepis_corone)),((((Dendrelaphis_seemani,Ingerophrynus_maculata),((Lamprolepis_mexicanum,Salmo_xanthocheilus),Nucifraga_dives)),Telescopus_wislizeni),Psammophis_carbonaria)),Thecla_avosetta)),((Chen_ocellatus,((Gambelia_avosetta,Hydrochelidon_blakistoni),Leiopython_spilota)),Lutra_fuellebornii))),(((((Atrophaneura_americanus,Middendorffinaia_tuberculosus),((Kinosternon_aristotelis,Lycodon_subniger),Picus_macqueni)),Osmoderma_baeri),Onychodactylus_vitulina),Spalax_pallidus)),((((((((Aythia_fimbriatus,(Castor_bairdi,(Phrynomerus_gratiosa,Sericinus_sudanensis))),Squaterola_iguana),((((Bronchocela_dentatus,Lampropeltis_taeniura),Diomedea_giganteus),(Kinixys_longicaudata,Xenopeltis_leporosum)),(Kinosternon_gebleri,Pandinus_melanuroides))),Capreolus_bicoloratum),Underwoodisaurus_bairdii),Eunectes_viridis),(Callipogon_pictus,((Cygnus_temminskii,Panthera_cristata),Pusa_monoceros))),Gazella_prasina)),Anser_comicus);
Squaterola_iguana Falcipennis_molurus

(((Acanthogonatus_japonica,Dipsosaurus_meles),Eryx_aureola),((((((((((((((((((((((Aegypius_minutus,Middendorffinaia_avocetta),(((Charadrius_vipio,Vulpes_vanellus),((((Epicrates_subcinctus,(Lycodon_weliczkowskii,Python_triangulum)),Lutra_vastus),Phrynohyas_erythrogastra),(Rhinolophus_spilota,Tiliqua_lutris))),Pusa_grandis)),((Dahurinaia_boulengeri,Scaphiopus_cavirostris),Porphyrio_caninus)),Aythya_nelsonii),Pterinochilus_pardus),Enhydris_korschun),(Gavia_adspersus,Gyps_hosii)),(Oceanodroma_equestris,Selenocosmia_plathyrhychos)),Dafila_pygmaeus),Alopex_altaica),Balaena_brongersmai),((Bos_agama,Hyla_aspera),Machetes_semipalmatus)),Apus_saxatilis),((((Callipogon_bimaculata,Oceanodroma_morinellus),Pareas_quinquestriatus),Grampus_bewickii),Tropidurus_trianguligerus)),Passer_scincoides),Meles_heliaca),(Heterodon_haliaetus,Melanocoryhpa_longipes)),Chrysopelea_leucopsis),Aplopeltura_bengkuluensis),Alloporus_clinatus),((Dafila_pica,Pseudemys_saxatilis),Phrynosoma_eximia)),(Heterodon_querquedula,Rhabdophis_hosii)),((((((((((((((((Alaus_grupus,Apalone_brongersmai),(Buthus_livia,Pterinochilus_politus)),Erpeton_glacialis),Chlamydotis_arvensis),(Trionyx_quinquetaeniata,Tylototriton_dendrophila)),((Haliaeetus_indica,Rhynchaspis_ichthyaetus),Larus_celeber)),Tiliqua_flavirufa),Rhinolophus_hosii),Enhydra_smaragdina),(Remiz_virgo,Triturus_guttifer)),(Chelydra_avinivi,(Mergus_ferina,Tetrao_livia))),(((((Bronchocela_melanoleucus,((Colaeus_geniculata,Hysterocrates_chinensis),(Lepus_tridactylum,Mergus_doriae))),(Castor_nyroca,(Circaetus_weberi,Teratolepis_linaria))),(Corytophanes_chrysargos,(Mustela_savignii,Osmoderma_clarus))),Corallus_ibera),Onychodactylus_mlokosiewiczi)),Chrysemys_wogura),(Anolis_limosa,Circaetus_exquisita)),(((((((Anolis_caudolineatus,Ruticilla_dione),(Clemmys_chukar,(Hadogenes_ameiva,Nyroca_karelini))),(Dendrobates_altaicus,(Ortigometra_decorus,Tringa_decor))),Apus_rudicolis),(Iguana_deserti,Ortigometra_flavigularis)),Trapelus_gallicus),Iomachus_perdix)),Eirenis_fuscatus));
Haliaeetus_indica Erpeton_glacialis

(Acanthoscurria_multifasciata,((((Acipenser_pugnax,Pratincola_albirostris),Underwoodisaurus_lehmanni),Scolopax_schokari),Cygnus_punctatus),((((((Actitis_ion,(((((Anolis_ulikovskii,(((Capella_terrestris,Osmoderma_vulpes),Oenanthe_aspera),Sturnus_franckii)),Salmo_leucocephala),Tursiops_hirundo),(Balaenoptera_albatrus,Megaloperdix_caucasicus)),Spalax_rosmarus)),((Capra_quadrivirgata,Telescopus_hungaricus),Ephibolus_altaica)),(Tadarida_eximia,Uncia_longipennis)),(((((((((((((((((Allobates_mandarina,Osteopilus_flavomaculatus),((Enhydra_subruficollis,Milvus_ruficollis),Motacilla_niloticus)),Falcipennis_aegyptia),Ceratophrys_lesueurii),Pterocles_lepturus),Chettussia_parahybana),((Epipedobates_viscivorus,Scorpio_squaterola),Motacilla_hasselquistii)),Bufo_caudata),(Ketupa_jaspidea,Saxicola_septentrionalis)),(Philacte_pallidus,Phrynocephalus_isabellina)),Spalerosophis_spilota),((Casarca_ruthveni,((Mareca_grossmani,Underwoodisaurus_noctua),(Neolycaena_veredus,Picus_calvus))),Saxicola_terrestris)),(Passer_cyanogaster,Streptopelia_belliana)),(Bronchocela_mutabilis,(Falco_alcinous,Lystrophis_deremensis))),Aythya_lividum),Bombina_albatrus),Iomachus_pictus)),(Passer_jubata,Rangifer_metallica)),((((((((Allobates_teniotis,(((((((Anas_fasciolata,Mareca_maihensis),(Eucratoscelus_ceterus,(Hyperoodon_pelagicus,Mabuya_erythrogastra))),Xenopeltis_riparia),Poephagus_dactylisonans),((Apalone_semipalmatus,Enhudra_novaeangliae),(((Bradypodion_naumanni,Holaspis_karelini),Camptoloma_modestus),Ptyodactylus_leucotus))),Tetraogallus_gibbosus),Mustela_getula)),Camptoloma_nigra),Eudrornias_hypoleucus),Eunectes_cristatus),(Hydrochelidon_onocrotalus,Picus_caryocatactes)),Porzana_blakistoni),Eulabeia_totanus),Gonyosoma_peregusna)));
Eulabeia_totanus Acanthoscurria_multifasciata

(,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,Furcifer_brevipes,,,,,,,,,,,,,,,Chlamydotis_vastus,,,,,,,,,,,,,,,,);
Furcifer_brevipes Chlamydotis_vastus

(((Acheron_cenchria,Dasypeltis_tigrinus),Milvus_bellii),(((((((((((((((((((Ameiva_pulchripes,((Chen_spinifera,Psammophis_subruficollis),Lamprophis_wogura)),(((Apalone_paradoxus,Regulus_holbrooki),((Prunella_lasiopterus,Tringa_nyroca),Terpsihone_isabellina)),Glareola_emilia)),Megaloperdix_armata),(Hirundo_rusticolus,Osteopilus_gecko)),Mustela_nivicola),Ceratophrys_yeltoniensis),(Gambelia_anatina,Nyroca_ferrumequinum)),Candoia_wislizeni),Elaphe_stagnalis),(Aythya_mexicanum,(Osteopilus_vipio,Saga_himantopus))),(Ptyodactylus_pusilla,Tetraogallus_circia)),Heterodon_insularis),(Ciconia_prominanus,Pelodiscus_mutabilis)),Riparia_flavomaculatus),Pelodiscus_rapax),Atrophaneura_laevis),Porphyrio_elaphus),(Bombina_mitratus,(Bombyx_chukar,(Capreolus_sauromates,((Charadrius_erythrogastra,Felis_arizonensis),Terpsihone_keyzerlingii))))),Athene_sauromates),((((((((((((Anthropoides_fiber,(Capeila_taxus,Rosalia_canorus)),((Charadrius_colombianus,(((Cygnus_dauricus,Scorpio_dominicus),Scaphiopus_fissipes),Pagophila_mexicana)),Minipterus_collybitus)),Pogona_bicoloratum),(((Balaena_ochropus,(Eublepharis_martensi,Platemys_leiosoma)),Megaptera_asperum),(Capeila_galactonotus,(Chelus_tetrix,Fulica_hodgsoni)))),Leuciscus_lehmanni),((Eirenis_cyanogaster,(Leptobrachium_adamsii,Xenophrys_caerulea)),Pelusios_tarda)),Diomedea_fuellebornii),(Limnodromus_boa,Tetraogallus_vipio)),((Callipogon_aeruginosus,Squaterola_blakistoni),Sphenurus_caucasicus)),Neolycaena_ibis),((((Chlidonias_schneideri,Gyps_perrotetii),Regulus_garullus),(((Gonocephalus_anachoreta,Leuciscus_torquata),Vipera_guttifer),Triturus_elegans)),(Eubalaena_caryocatactes,Numenius_cioides))),((Aquila_mnemosyne,Neolycaena_cherrug),(Pica_unicolor,Synthliboramphus_subminiatus))));
Xenophrys_caerulea Tringa_nyroca

((((((((((((((((((Acanthis_platyrhinos,(((Apalone_baibacina,Pterinochilus_flavigularis),((((Branta_wislizeni,((Litoria_fulvus,Phormictopus_amethistina),Otis_gebleri)),Geochelone_deserti),Leiocephalus_mugodjaricus),(((((Himantopus_resinifictrix,Pagophila_flavolineata),((Petrocincla_cliffordii,Sitta_piceus),Phrynomerus_chrysaetos)),(Rosalia_temminskii,Tiliqua_acuta)),Lyrurus_scrofa),Opheodrys_venulosa))),(Chrysopelea_cornix,Opheodrys_albatrus))),Pelomedusa_ferox),Canis_spinifera),Salamandra_vitticeps),((((Anas_ibera,(Querquedula_latiscutatus,Telescopus_uluguruensis)),Coregonus_gregaria),(Chrttusia_melleri,Plegadis_lasiopterus)),((Anser_govinda,((Bradyporus_chrysaetus,Gerrhosaurus_boulengeri),(Homalopsis_leucoryphus,Polypedates_dives))),Physignathus_vitticeps))),(((Grammostola_medici,Lutra_cyanus),Middendorffinaia_erythropus),Thymallus_albatrus)),Charadrius_latiscutatus),((((Ahaetulla_altaicus,Eumeces_pygargus),Circaetus_avicularia),Oligodon_bonasus),Haplopelma_crucigera)),Perdix_nigriceps),(((Alpes_niloticus,Pterinochilus_helvetica),Xenochrophis_dulkeitiana),Petrocincla_ferrumequinum)),(((((Alopex_alpina,(Callipogon_leucocephala,Scincus_totanus)),(((Dendrelaphis_monedula,(Oligodon_crispus,Pseudorca_boyciana)),(Laudakia_boulengeri,Philothamnus_odoratus)),Paraphysa_communis)),Pelusios_calyptratus),Cinclus_bairdii),Fuligula_proteus)),(Epipedobates_cliffordii,(Gallinago_boschas,Synthliboramphus_Bernicla))),Hadrurus_dignus),((((((((Chrysopelea_crassicauda,Holodactylus_atriceps),(Ninox_reinwardti,((Pedostibes_dexter,Phasianus_leucotus),Sturnus_casualis))),(((Lamprolepis_franckii,Rhesus_nigropalmatus),Salvelinus_scripta),Micropalama_moschata)),Pterinochilus_dignus),Mylopharyngodon_uncia),Ruticilla_pholeter),Scolopax_venulosa),(Colaeus_salei,(Gyps_caesius,Vormela_iankowskii)))),Pseudorca_variabilis),Nipponia_onocrotalus),Enhydris_macrops),(((Acheron_parreyssi,Tiliqua_carnivorus),(Elseya_rufodorsata,((Enhudra_cristata,(Myotis_madagascariensis,(Sorex_javanica,Spizaetus_dexter))),Pandinus_thibetanus))),(Eucratoscelus_cinclus,Sphenurus_plumifrons)),Aegialifes_seemani);
Ahaetulla_altaicus Oligodon_bonasus

(,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,Eudramias_pedo,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,sibiricus_avosetta);
Eudramias_pedo sibiricus_avosetta

((Acanthoceros_americanus,Camptoloma_erythronota),(((Acheron_cachinans,Rhacodactylus_helvetica),Paradoxornis_striatus),(((((((((Alauda_clericalis,Dyscophus_torquatus),((((((Cervus_colchicus,(Cypselus_getula,Epipedobates_pelagicus)),(((Chen_corone,(Cynops_hemionus,Monodon_diffidens)),Iomachus_serricollis),(Dipsosaurus_glutinosus,Gyps_hodgsoni))),((Erpeton_clarus,Nucifraga_plumifrons),Phormictopus_atthis)),Melanocoryhpa_infrafrenata),(Ctenosaura_elegans,(Turdus_ocellatus,Underwoodisaurus_leucostomum))),(Gonocephalus_uluguruensis,(Rhabdophis_papuana,Vulpes_odoratus)))),(((((((((((((((Alloporus_arseniavi,(Androctonus_kopsteini,((Kinixys_garulla,Synthliboramphus_sieboldii),(Phalaropus_maritimus,Phrynops_heudei)))),Eurynorhynchus_euptilura),Colaeus_arenarius),Himantopus_aestivus),Eubalaena_mystaceus),Tylototriton_hipposideros),(((Argynnis_heudei,(Buteo_avinivi,(Latastia_nasuta,Teratolepis_avinivi))),(((((Chen_rufus,(Fuligula_monilis,Mylopharyngodon_bicoloratum)),Hadrurus_sagrei),((Circus_brevipes,(Colaeus_cepediana,Euspiza_radiata)),Lagenorhynchus_czerskii)),(Falcipennis_aristotelis,Gongylophis_erythronotus)),Uncia_tinctorius)),(Myotis_interiorata,Sphenurus_gregarius))),Leuciscus_carinata),Anthropoides_chuatsi),Pratincola_nipalensis),Clemmys_pygmeus),((Chrttusia_turtur,(Gerrhosaurus_leucoptera,(Phalacrocorax_vereda,(Sericinus_weliczkowskii,Spermophilus_sinensis)))),Diomedea_miliaris)),Elaphe_bobac),Scolopendra_purpurascens),Porphyrio_enydris)),Rhinolophus_cristatellus),Limnaeus_coelestis),(Bronchocela_belliana,(((((Cypselus_leptochelis,Psalmopoeus_moschiferus),Epipedobates_pedo),(Eublepharis_cocincinus,Rhombomys_sirtalis)),Gongylophis_altaica),Dasypeltis_milii))),Chelodina_lopatini),Salvelinus_drapiezii),(Phormictopus_rufodorsata,Triturus_plumifrons))),((Actitis_boschas,Net_medirostris),Hydrochelidon_dignus));
Gongylophis_erythronotus Eubalaena_mystaceus

(,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,Coregonus_similis,,,,,,,,,,,,,,,,,,,,Alcedo_siebenrocki,,,,);
Coregonus_similis Alcedo_siebenrocki

((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((())Ziphius_erythropus)))))))))))))Sternotherus_flavomaculatus))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))));
Ziphius_erythropus Sternotherus_flavomaculatus

(Zosterops_gigas,Uncia_monedula,Tylototriton_torquatus,Tylototriton_lutra,Triturus_metallica,Trachemys_leucogaster,Theloderma_hemionus,Thamnophis_asperum,Terpsihone_smithi,Teratolepis_naumanni,Teratolepis_ferrumequinum,Telescopus_vertebralis,Synthliboramphus_caeruleus,Sus_erythropus,Spalax_helvetica,Scolopax_calamita,Rhamphiophis_sauromates,Rhabdophis_schokari,Rhabdophis_mehelyi,Pseudemys_erythropus,Pratincola_ceterus,Porphyrio_turtor,Porphyrio_glutinosus,Phrynops_colubrinus,Philomachus_aureostriata,Phalacrocorax_zenobia,Parnassius_saiga,Parnassius_glareola,Paraphysa_emilia,Parabuthus_microlepis,Ovis_hirundo,Otis_sagrei,Nyctaalus_exanthematicus,Norops_nivalis,Nhandu_maihensis,Nemachilus_sanguinolentus,Monodon_ladogensis,Megaptera_tristis,Macrorhamphus_ferruginea,Macrorhamphus_crucigera,Machetes_tataricus,Limosa_carolinensis,Leiolepis_medici,Kaloula_heliaca,Hyla_leucogaster,Gonyosoma_cliffordii,Gekko_pygmaeus,Gecarcinus_bimaculata,Gambelia_belliana,Eutamias_sagrei,Euspiza_mystaceus,Eudramias_catenifer,Eirenis_hirundo,Diomedea_javanica,Desnana_leucorodia,Cygnus_keyzerlingii,Cyclagras_godlewskii,Ctenotus_hodgsoni,Citellus_barroni,Cinclus_lavaretus,Chrysemys_altaicus,Chondropython_bedriagai,Chlidonias_aureostriata,Chamaeleo_pallidus,Carabus_sibilans,Carabus_platyrhinos,Candoia_iguana,Bronchocela_tatarica,Branta_flava,Brachyramphus_lutris,Bos_medirostris,Bombyx_stejnegeri,Bombycilla_constricticollis,Boa_hassanica,Basiliscus_cepediana,Balaena_barbatus,Babycurus_oedicnemus,Aplopeltura_riparia,Androctonus_falcinellus,Anas_taczanowskius,Allactaga_gobio,Alectoris_tenuirostris,Alaus_kingii)Zosterops_nippon;
Branta_flava Porphyrio_turtor

(Alloporus_opimus,(((((Anolis_wislizeni,Desnana_trianguligerus),(((((((((((((((Brachypelma_belliana,Ursus_viscivorus),((Haliaeetus_lasiopterus,Pelecanus_vitticeps),Motacilla_conicus)),Tropidurus_leschenaultii),((((((Chlamydotis_ovata,Pelusios_dentatus),Testudo_leiosoma),Melanocorypha_penelope),Eudramias_middendorffi),Haplopelma_bewickii),Grampus_lepturus)),Camptoloma_smaragdina),(Leuciscus_martensi,(Megophrys_mehelyi,Pyxicephalus_maurus))),(Dendrobates_docilis,(((((Kassina_ruthveni,(Micropalama_nigropalmatus,Trionyx_vegans)),Sphenurus_eulophotes),Lamprophis_galactonotus),Megaloperdix_peregusna),Leiopython_cygnoides))),((Bronchocela_morinellus,Eubalaena_zenobia),(Mergus_hosii,Phalacrocorax_caerulea))),Mergus_aleutica),Megaloperdix_scincus),(Minipterus_zagrosensis,Synthliboramphus_quinquestriatus)),((Ephibolus_albicilla,(Grampus_coelestis,Ortigometra_subrufa)),Eublepharis_calamita)),(Bufo_fulvus,Epipedobates_pyromelana)),((((((((((Bronchocela_geniculata,Pachytriton_longipennis),(Eumeces_quadrivirgata,Rhynchophis_citrsola)),(Bronchocela_microlepis,Lamprolepis_unicolor)),Salmo_ichthyaetus),(Iomachus_longicollis,Scolopendra_hypomelus)),Lepus_mexicanum),(Diomedea_fiber,(Macrorhamphus_subrufa,Megaptera_latiscutatus))),Camptoloma_crocodilus),Lasiodora_terrestris),Physignathus_cenchria)),((((((((((Capreolus_reinwardti,Grus_siebenrocki),(((Chrysopelea_garmani,Pelusios_monacha),Ursus_jaculus),Paraphysa_castus)),Gavia_difficilis),Rhynchophis_bellii),Middendorffinaia_strepera),Pelusios_rutila),Crotaphytus_daurica),(((((((Cinclus_rufus,Oceanodroma_bicoloratum),Phalaropus_aureola),Haliaetus_squamatus),Sericinus_nigriceps),(((Grampus_citrsola,Spalerosophis_ammon),(Nyctixalus_atthis,Osmoderma_percnopterus)),Turdus_decorus)),Sturnus_hongkongensis),(Netta_albicilla,Tadarida_aceras))),Tiliqua_bifasciatus),Cuon_undulata))),Enhudra_septentrionalis),(Argynnis_mitratus,(Gonocephalus_plumipes,Remiz_paradisi))),Elseya_pugnax),((((((((Aquila_hendricksoni,Triturus_lagopus),Chamaeleo_flammea),Hydrosaurus_alterna),Pyrrhocorax_soloensis),Thymallus_keyzerlingii),Eulabeia_borealis),Moschus_marmoratus),Capra_dominus));
Megaloperdix_peregusna Chlamydotis_ovata

(((((((((((((((((((((,),),),),),(((((,),(,)),),),)),),),((((((,((,(((,(,)),),)),(((((,(,)),(((,),),)),),Fulica_piscator),(((((,),),),),)))),(((((,),),(,)),),)),),),),)),),((,(,)),)),),),),),(((,),),)),((,),(,))),((((((((((((,(,((,),((,),)))),),),),),),Meles_percnopterus),),),),(,)),)),),),(,),(((,),),(,)));
Meles_percnopterus Fulica_piscator

(Aegypius_grossmani,Allobates_scincoides,Alopex_nyroca,Apodora_penelope,Archispirostreptus_enydris,Argynnis_bengkuluensis,Athene_reticulatus,Aythia_bobac,Balaenoptera_maritimus,Balaenoptera_truncatus,Bombyx_nivalis,Bradypterus_guttifer,Bufo_falcinellus,Burhinus_fluviatilis,Caiman_sauritus,Calidris_dulkeitiana,Callipogon_cyanogaster,Cervus_hosii,Cervus_wislizeni,Chen_gobio,Chrysemys_davidiana,Cinclus_personata,Circus_dominicus,Coregonus_alcinous,Cyclemys_leucotus,Cygnus_subglobosa,Dafila_docilis,Dendrelaphis_cinaedus,Dyscophus_javanica,Eirenis_guentheri,Gambelia_nipalensis,Gekko_leucomystax,Hadogenes_auriculatus,Haliaetus_irregularis,Haplopelma_pachypus,Hirundo_sauritus,Hydrochelidon_nipalensis,Lamprolepis_cristatellus,Lasiodora_longicollis,Leiolepis_nigrolineatus,Leiopython_galactonotus,Leiopython_latiscutatus,Leptobrachium_himantopus,Lepus_tigrinus,Limnodromus_dexter,Limosa_adamsii,Melanocorypha_cliffordii,Mylopharyngodon_nupta,Mylopharyngodon_pachypus,Norops_euptilura,Norops_marcianus,Notophthalmus_marmoratus,Numenius_caryocatactes,Odonthurus_tentaculatum,Oedura_kraepelini,Otis_carinata,Paradoxornis_mexicana,Paraphysa_laevis,Pareas_caudicinctus,Pelomedusa_mexicana,Pelusios_dendrophila,Phormictopus_nigra,Phrynops_parvus,Polypedates_calamita,Procellaria_euptilura,Ptyodactylus_coturnix,Ptyodactylus_hypoleucus,Pusa_cioides,Pyrrhocorax_longicollis,Querquedula_jaspidea,Recurvirostra_boulengeri,Rhacodactylus_cavimanus,Rhacodactylus_chrysaetus,Rhesus_coloratovillosum,Rhinolophus_grus,Scincus_novaeangliae,Scorpio_nivalis,Telescopus_subglobosa,Tetraogallus_salvator,Tringa_apus,Tupinambus_regius,Tupinambus_squamatus,Uncia_carinata,Underwoodisaurus_altaica,Upupa_temminckii,Uromastyx_collaris,Vulpanser_cranwelli,Xenophrys_uluguruensis,Ziphius_leiosoma,Zosterops_vulpes)Acipenser_azureus;
Alopex_nyroca Paradoxornis_mexicana

(((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((()))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))Chrysopelea_cambridgei)))Chalcides_aceras))))))))))))));
Chalcides_aceras Chrysopelea_cambridgei

(((((Accipiter_ameiva,(Cinclus_agama,Pusa_graeca)),(((((((((((((Archispirostreptus_penelope,Monticola_canorus),Salmo_squamatus),(((Dyscophus_aegyptia,Eunectes_politus),Iguana_hodgsoni),Pelomedusa_elegans)),Limosa_tentaculatum),(((Chlamydotis_maldivarum,Lampropeltis_mykiss),(Heteroscodra_sujfunensis,Nyctixalus_elegans)),Osmoderma_ferruginea)),Chlamydosaurus_eremita),Oxyura_tridactylum),(((Ctenosaura_terrestris,Phalacrocorax_kazanakowi),((Cygnopsis_punctatus,(Gazella_rufus,Mylopharyngodon_pulchra)),Xenophrys_personata)),((Eucratoscelus_chuatsi,Ptyodactylus_ladogensis),Phylloscopus_pugnax))),Ketupa_bifasciatus),Tylototriton_mehelyi),Spizaetus_erythronota),Spalax_glaucescens),(Archispirostreptus_pugatshuki,Homalopsis_paradisi))),(Homopholis_scripta,Recurvirostra_albigula)),((Avicularia_falcipennis,Cottus_blythi),Thamnophis_docilis)),((((((Alloporus_angustirostris,Lanius_tarandus),Thymallus_ferruginea),Oedura_dulkeitiana),Ardea_reticulatus),(Ambystoma_hendricksoni,(Basiliscus_emilia,((Hadrurus_leucogeranus,((Iomachus_standingii,Vulpes_flava),(Scolopendra_politus,Sorex_linaria))),Oceanodroma_sieboldii)))),Citharacanthus_hungaricus),(((((((((((((((((Ambystoma_albocinctus,(Anthropoidae_quadrivirgata,(((((Bronchocela_deminutus,Nemorhaedus_castaneus),(((Cottus_torquata,(Homopholis_gebleri,Ptychozoon_infrafrenata)),Eurynorhynchus_calvus),Gypaetus_saxatilis)),Gecarcinus_mongolica),Sceloporus_elaphus),Salvelinus_anatina))),(Aphonopelma_ruthveni,Hyla_calvus)),Balaenoptera_fluviatilis),Melanocorypha_collybitus),Riparia_gratiosa),Chlidonias_longipennis),Ardea_hyemalis),Pachydactylus_vereda),(Furcifer_relictus,Gallinago_nivalis)),Chen_hilarii),(Avicularia_macrops,Rhesus_dulkeitiana)),(Lobipes_helena,Milvus_vitticeps)),Budytes_colubrinus),((Ameiva_adspersus,((((Amphiuma_medirostris,(Dasypeltis_sphenocercus,Sitta_vittatus)),(Chlamydotis_crocodilus,Eunectes_fuscatus)),Hydrosaurus_clinatus),Pratincola_ignicapillus)),Branta_savignii)),Bufo_davidiana),Underwoodisaurus_galeatus),(Buthacus_lineatus,((Coturnix_cyanea,(Melanocorypha_mirabilis,Mesoplodon_blakistoni)),(Cygnopsis_compactus,Euspiza_marcianus)))));
Pusa_graeca Cottus_torquata

(((((((Alauda_blythi,Eunectes_taxus),Platemys_decor),((((((((Bufo_govinda,Remiz_intermedia),Procellaria_piceus),Grus_schreibersi),Motacilla_atra),Cyriopagopus_flava),Heterodon_macularius),((Capreolus_flavescens,Eulabeia_colchicus),Enhydra_pugnax)),(Coturnix_hyperboreus,Opheodrys_tatarica))),Physignathus_teguixin),Gonocephalus_caerulea),Phalaropus_septentrionalis),((Alaus_giganteus,(Milvus_canus,Querquedula_filipjevi)),(Lycodon_gregarius,Zosterops_mehelyi)),(((((((((Alopex_lepturus,Ephibolus_azureus),Gyps_multituberculatus),(((((Arenaria_taxus,Buthus_alcinous),((Middendorffinaia_mandarina,Monodon_seemani),Scorpio_casualis)),Platemys_rufina),(Nerodia_diffidens,(Rhodostethia_xanthocheilus,(Spizaetus_fuliginosus,Sterna_ceterus)))),((Bombus_plumifrons,Chrysemys_turtor),Mareca_leschenaultii))),Hydrosaurus_altaica),((Ameiva_argentatus,Dafila_scincus),Ciconia_nivicola)),Lycodon_lagopus),(((((((Anolis_viridis,Anthropoidae_nivicola),Calidris_aureola),(Eutamias_bengalensis,Glareola_argali)),Spalerosophis_cyanochloris),Aquila_modestus),Haplopelma_rhymnus),(Ctenotus_serpentina,Eucratoscelus_emarginatus))),((((((((((((Apalone_caeruleus,Pachytriton_japonensis),(Fulica_hypoleucus,(((Lamprophis_leucotus,Nhandu_hodgsoni),Rosalia_dennysii),sibiricus_govinda))),Sterna_collectivus),Synthliboramphus_eulophotes),Bufo_cliffordii),Querquedula_cinclus),Thymallus_ridibundus),(Emberiza_schneideri,Pyrgilauda_linaria)),Megaloperdix_flavomaculatus),Apodora_geniculata),((Avicularia_nyroca,(((Branta_jaculus,Tryngites_lasiopterus),(Eirenis_citrsola,Telescopus_exquisita)),(Phyllopneuste_salamandra,Pusa_glacialis))),((((Cinclus_plumifrons,(Elaphe_sanguinolentus,(Pedostibes_parreyssi,Phylloscopus_holbrooki))),(((Clemmys_nebularia,Felis_musculus),Prunella_licin),Pelodytes_hispida)),Phalacrocorax_leuconotus),Glareola_erythronotus))),((Balaenoptera_ochropus,(((Ethmostigmus_novaeangliae,Leuciscus_martensi),Glareola_leporosum),Porphyrio_paradisi)),Leptopelis_variabilis))),Thamnophis_gregarius));
Remiz_intermedia Heterodon_macularius

((((Acanthosaura_carbonaria,Net_fischeri),((((((((Aythia_longipennis,(Callipogon_minor,((Cyclemys_tinnunculus,Phrynocephalus_leucogaster),Emydura_physalus))),Chlamydotis_guttifer),Cyriopagopus_striatus),Phelsuma_crispus),Ziphius_scincus),Cyriopagopus_epops),Norops_sp),Scaphiopus_bimaculata)),Scaphiophryne_garullus),Aix_nippon,((((((((((Aix_pygmaeus,Scaphiophryne_stellatum),Philothamnus_anser),((((((((Alcedo_communis,Sterna_alterna),Bradypterus_bengalensis),Kinixys_barroni),Seokia_maculatum),Gonocephalus_bewickii),Circaetus_rufina),Zosterops_clarus),((((((Anas_jubata,(Camptoloma_martensi,Teratoscincus_korschun)),((Gallinago_salamandra,(Hottentotta_vertebralis,Rissa_garmani)),Grammostola_wogura)),(Enhydris_mitratus,Gambelia_cocincinus)),((Phalacrocorax_salei,Platemys_carinatus),Sphenurus_paradoxus)),Bos_bicoloratum),Rhacodactylus_totanus))),Ambystoma_altaicus),((((Certhia_avocetta,Larus_mackloti),(Homopholis_japonensis,Vormela_gebleri)),(Hyla_variabilis,Leiocephalus_auriculatus)),((Chrttusia_melanostictus,Limnaeus_colombianus),Platalea_sepsoides))),(Cinclus_emarginatus,Rangifer_decor)),(((((((((Ameiva_lasiopterus,Cuora_imperator),Heterodon_japonensis),(((((((Androctonus_tetrax,Burhinus_ochropus),Mesoplodon_longipes),Rhabdophis_flammea),(Litoria_cianeus,Parnassius_colubrinus)),Boiga_unicus),Capra_paradoxus),((((Gypaetus_paradisi,(Net_guttata,Telescopus_melanuroides)),Ruticilla_scripta),(Nemorhaedus_albatrus,(Paradoxornis_rosmarus,Tiliqua_valliceps))),(Lagenorhynchus_falcinellus,Philomachus_fuscus)))),Passer_plathyrhychos),Otocoris_dauricus),Phrynocephalus_subrufa),Eudrornias_hyperboreus),Cervus_doctus),Trapelus_japonica)),Hyperoodon_hendersoni),Leiurus_anatina),Eschrichtius_placidus));
Circaetus_rufina Otocoris_dauricus
"""

In [ ]:
print(get_answer(input_txt))